In [12]:

from pyproj import Transformer
import math

# Configuration
GRID_SIZE = 10_000  # 10 km
SOURCE_CRS = "EPSG:4326"
TARGET_CRS = "EPSG:3857"

# Transformer: WGS84 -> EPSG:6933
transformer = Transformer.from_crs(
    SOURCE_CRS,
    TARGET_CRS,
    always_xy=True
)

def get_grid(lat: float, lon: float) -> dict:
    """
    Convert latitude/longitude to EPSG:6933
    and determine the corresponding 10km grid.
    """

    # Step 1: Transform coordinates
    # always_xy=True => input order is lon, lat
    x, y = transformer.transform(lon, lat)

    # Step 2: Calculate grid indices
    col = math.floor(x / GRID_SIZE)
    row = math.floor(y / GRID_SIZE)

    # Step 3: Calculate grid boundaries
    xmin = col * GRID_SIZE
    xmax = (col + 1) * GRID_SIZE

    ymin = row * GRID_SIZE
    ymax = (row + 1) * GRID_SIZE

    # Step 4: Generate grid ID
    grid_id = f"E6933_10K_R{row}_C{col}"

    return {
        "grid_id": grid_id,
        "row": row,
        "col": col,
        "x": x,
        "y": y,
        "xmin": xmin,
        "xmax": xmax,
        "ymin": ymin,
        "ymax": ymax,
    }


# Example
lat = 10.790337
lon = 106.704071

result = get_grid(lat, lon)

for key, value in result.items():
    print(f"{key}: {value}")


grid_id: E6933_10K_R120_C1187
row: 120
col: 1187
x: 11878242.84928931
y: 1208338.7827616634
xmin: 11870000
xmax: 11880000
ymin: 1200000
ymax: 1210000


In [ ]:
import math, os, json, textwrap

# Input coordinate: latitude, longitude
lat = 10.790337
lon = 106.704071
R = 6378137.0
GRID_SIZE = 10_000

# Forward Web Mercator
phi = math.radians(lat)
lam = math.radians(lon)
x = R * lam
y = R * math.log(math.tan(math.pi / 4 + phi / 2))

col = math.floor(x / GRID_SIZE)
row = math.floor(y / GRID_SIZE)

xmin = col * GRID_SIZE
xmax = (col + 1) * GRID_SIZE
ymin = row * GRID_SIZE
ymax = (row + 1) * GRID_SIZE

def inverse_3857(px, py):
    out_lon = math.degrees(px / R)
    out_lat = math.degrees(2 * math.atan(math.exp(py / R)) - math.pi / 2)
    return [out_lat, out_lon]

# Leaflet expects [lat, lon]
polygon = [
    inverse_3857(xmin, ymin),
    inverse_3857(xmax, ymin),
    inverse_3857(xmax, ymax),
    inverse_3857(xmin, ymax),
]

grid_id = f"E3857_10K_R{row}_C{col}"
center_lat, center_lon = inverse_3857((xmin+xmax)/2, (ymin+ymax)/2)

html = f"""<!DOCTYPE html>
<html lang="vi">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Fire Grid 10 km - EPSG:3857</title>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
        integrity="sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=" crossorigin=""/>
  <style>
    html, body {{ height: 100%; margin: 0; font-family: Arial, sans-serif; }}
    #map {{ height: 100%; width: 100%; }}
    .info {{
      position: absolute; top: 14px; left: 54px; z-index: 1000;
      background: white; padding: 12px 14px; border-radius: 9px;
      box-shadow: 0 1px 8px rgba(0,0,0,.25); max-width: 310px;
      line-height: 1.45; font-size: 13px;
    }}
    .info strong {{ font-size: 15px; }}
    .legend {{ margin-top: 7px; color: #444; }}
    .swatch {{ display:inline-block; width:13px; height:13px; margin-right:6px;
               vertical-align:-2px; border:2px solid #e06b22; background:rgba(255,145,45,.22); }}
    @media (max-width: 520px) {{
      .info {{ left: 54px; right: 10px; max-width: none; font-size: 12px; }}
    }}
  </style>
</head>
<body>
<div id="map"></div>
<div class="info">
  <strong>{grid_id}</strong><br>
  CRS: EPSG:3857 · Grid: 10,000 × 10,000 m<br>
  Point: {lat:.5f}, {lon:.5f}<br>
  X: {x:,.3f} m · Y: {y:,.3f} m<br>
  Row: {row} · Col: {col}
  <div class="legend"><span class="swatch"></span>Ô grid chứa điểm cháy</div>
</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"
        integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=" crossorigin=""></script>
<script>
  const map = L.map('map');
  L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
    maxZoom: 19,
    attribution: '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
  }}).addTo(map);

  const point = [{lat}, {lon}];
  const grid = {json.dumps(polygon)};

  const cell = L.polygon(grid, {{
    color: '#e06b22',
    weight: 3,
    fillColor: '#ff9b45',
    fillOpacity: 0.22
  }}).addTo(map);

  cell.bindPopup(
    '<b>{grid_id}</b><br>Grid 10 km × 10 km trên EPSG:3857<br>' +
    'X: [{xmin:,}, {xmax:,}) m<br>Y: [{ymin:,}, {ymax:,}) m'
  );

  const marker = L.marker(point).addTo(map);
  marker.bindPopup(
    '<b>Điểm đầu vào</b><br>Latitude: {lat}<br>Longitude: {lon}<br>' +
    'FRP: chưa cung cấp'
  ).openPopup();

  L.control.scale({{metric: true, imperial: false}}).addTo(map);
  map.fitBounds(cell.getBounds().pad(0.30));
</script>
</body>
</html>
"""

out_path = "fire_grid_epsg3857_map.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Đã tạo bản đồ: {out_path}")
print(f"Grid ID: {grid_id}")
print(f"EPSG:3857 X={x:.3f}, Y={y:.3f}")
print("Các đỉnh grid [lat, lon]:")
for p in polygon:
    print(p)


Đã tạo bản đồ: fire_grid_epsg3857_map.html
Grid ID: E3857_10K_R120_C1187
EPSG:3857 X=11878242.849, Y=1208338.783
Các đỉnh grid [lat, lon]:
[10.716743924325923, 106.6300242249872]
[10.716743924325923, 106.71985575339914]
[10.804995755393525, 106.71985575339914]
[10.804995755393525, 106.6300242249872]
